In [ ]:
#These are the cells we have been running in this order for Auto-NBA to work

#The original notebook i used has about 30-40 more troubleshooting cells, but this combination seems to work the most reliably

#Most issues are with dependencies and the sed command are sed to bypass broken imports with better versions
#finding the correct libraries to use instead took a lot of troubleshooting, but these commands are the ones that reliably work

#i had to google/research the sed and grep commands as well as these libraries as I was not familiar with them


#clone into repo
!git clone https://github.com/RICE-EIC/Auto-NBA.git
%cd Auto-NBA

Cloning into 'Auto-NBA'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 30 (delta 2), reused 30 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 711.38 KiB | 2.42 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Auto-NBA


In [ ]:
#install libraries - thop gives a lot of issues
!pip install easydict
!pip install torchcontrib
!pip install thop

  Preparing metadata (setup.py) ... done
  Created wheel for torchcontrib: filename=torchcontrib-0.0.2-py3-none-any.whl size=7516 sha256=5623eb0f0914f83ccab29e9fc080b3f52e19718aa41541b99d14cda301b3dc68
  Stored in directory: /root/.cache/pip/wheels/e3/d1/1f/63f00ffea223db446943147a04ff035eb40d00cec3e87d63e5
Successfully built torchcontrib


In [ ]:
#swap the count hooks out as it give issues

!sed -i 's/from thop.count_hooks import count_convNd/from thop.vision.basic_hooks import count_convNd/' operations.py
!sed -i 's/from thop.count_hooks import count_convNd/from thop.vision.basic_hooks import count_convNd/' train_search.py

In [ ]:
#swap the syntax of the dataloader call as this also throws issues

!sed -i 's/dataloader_model.next()/next(dataloader_model)/' train_search.py
!sed -i 's/dataloader_arch.next()/next(dataloader_arch)/' train_search.py

In [ ]:
#another install
!pip install tensorboardX

#if config files have not been changed at this point, now is when to change them

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.1 MB/s eta 0:00:00


In [ ]:
#run the auto-nba search. should not throw errors now but still does infrequently
#prior commands take care of most issues for training search
#did not do a full run in this specific notebook hence the output being incomplete and wrong number of epochs
#I just copied the cells over so this notebook wouldnt have all the uneccesary troubleshooting cells and then tested cells to make sure they work

#need to be on gpu
!python train_search.py

04/19 05:41:49 PM args = {'seed': 12345, 'repo_name': 'Auto-NBA', 'world_size': 1, 'multiprocessing_distributed': False, 'rank': 0, 'dist_backend': 'nccl', 'dist_url': 'tcp://eic-2019gpu5.ece.rice.edu:10001', 'gpu': None, 'dataset': 'cifar100', 'dataset_path': '/home/yf22/dataset/', 'num_classes': 100, 'num_train_imgs': 50000, 'num_eval_imgs': 10000, 'bn_eps': 1e-05, 'bn_momentum': 0.1, 'opt': 'Sgd', 'momentum': 0.9, 'weight_decay': 0.0005, 'betas': (0.5, 0.999), 'num_workers': 8, 'grad_clip': 5, 'pretrain': 'ckpt/search', 'num_layer_list': [1, 4, 4, 4, 4, 4, 1], 'num_channel_list': [16, 24, 32, 64, 112, 184, 352], 'stride_list': [1, 1, 2, 2, 1, 2, 1], 'stem_channel': 16, 'header_channel': 1504, 'enable_skip': True, 'num_bits_list': [4, 6, 8], 'early_stop_by_skip': False, 'perturb_alpha': False, 'epsilon_alpha': 0.3, 'optim_mode': 'bilevel', 'train_mode': 'iterwise', 'sample_func': 'gumbel_softmax', 'temp_init': 5, 'temp_decay': 0.975, 'mode': 'proxy_hard', 'hard': True, 'offset': True

In [ ]:
#start of cells for training
#have to install a specific thop version for training or else everything breaks

!pip uninstall thop -y
!pip install thop==0.0.31-2005241907

Found existing installation: thop 0.1.1-2209072238
Uninstalling thop-0.1.1-2209072238:
  Successfully uninstalled thop-0.1.1-2209072238


In [ ]:
#change the iterable import
!sed -i 's/from collections import Iterable/from collections.abc import Iterable/g' \
/usr/local/lib/python3.12/dist-packages/thop/utils.py

#change the thop hooks thing again but in the training file
!sed -i 's/from thop.count_hooks import count_convNd/from thop.vision.basic_hooks import count_convNd/g' \
/content/Auto-NBA/train.py

#removes any remaing thop issues
!grep -rl "thop.count_hooks" /content/Auto-NBA | xargs sed -i 's/from thop.count_hooks import count_convNd/from thop.vision.basic_hooks import count_convNd/g'

In [ ]:
#run training which will also output best accuracy
#no output as i cant test training without first completing a search in this specific notebook but this exact command sequence is what was in the messier notebook and has been working
!python train.py